# 05 — Backtest Validation
Runs a 30-day held-out backtest of the trained DeepScalper models using **Lumibot's
built-in backtester**, then generates:
- A performance report (total return, Sharpe, max drawdown, win rate).
- An equity curve chart vs. SPY buy-and-hold.

**Input:**  `/content/drive/MyDrive/algo_trader/weights/{TICKER}.pth`  
**Period:** Most recent 30 days of bar data (held out from training).

> The backtester uses `YahooDataBacktesting` as the free data source.  
> Switch to `PandasDataBacktesting` with your Alpaca parquets for full accuracy.

In [ ]:
!pip install -q lumibot alpaca-py torch pyarrow pandas matplotlib tqdm python-dotenv

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys
WEIGHTS_DIR = '/content/drive/MyDrive/algo_trader/weights'
REPO_DIR    = '/content/deepscalper_copilot'

if REPO_DIR + '/algo_trader' not in sys.path:
    sys.path.insert(0, REPO_DIR + '/algo_trader')

ALPACA_API_KEY    = userdata.get('ALPACA_API_KEY')
ALPACA_SECRET_KEY = userdata.get('ALPACA_SECRET_KEY')
print('Credentials loaded ✓')

In [ ]:
REPO_URL = 'https://github.com/YOUR_GITHUB_USERNAME/deepscalper_copilot.git'
if not os.path.exists(REPO_DIR + '/.git'):
    !git clone {REPO_URL} {REPO_DIR}
print('Repo ready ✓')

In [ ]:
# Patch weights directory so strategy.py loads from Drive
import os
os.environ['WEIGHTS_DIR_OVERRIDE'] = WEIGHTS_DIR
os.environ['ALPACA_API_KEY']       = ALPACA_API_KEY
os.environ['ALPACA_SECRET_KEY']    = ALPACA_SECRET_KEY

In [ ]:
from datetime import date, timedelta
from lumibot.backtesting import YahooDataBacktesting

# Import the live strategy (it will load weights from WEIGHTS_DIR_OVERRIDE)
from execution.strategy import MultiStockDeepScalper
from execution.broker import get_broker

# 30-day held-out period (adjust end to today)
BACKTEST_END   = date.today() - timedelta(days=1)
BACKTEST_START = BACKTEST_END - timedelta(days=30)

print(f'Backtest period: {BACKTEST_START} → {BACKTEST_END}')

results = MultiStockDeepScalper.backtest(
    YahooDataBacktesting,
    BACKTEST_START,
    BACKTEST_END,
    benchmark_asset='SPY',
    parameters={},
    show_plot=False,          # We'll render the plot ourselves below
    show_tearsheet=False,
    save_tearsheet=True,
    tearsheet_file='/content/backtest_tearsheet.html',
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ── Key metrics ─────────────────────────────────────────────────────────────
metrics = {
    'Total Return (%)' : round(results.get('total_return', 0) * 100, 2),
    'Sharpe Ratio'     : round(results.get('sharpe_ratio', 0), 3),
    'Max Drawdown (%)'  : round(results.get('max_drawdown', 0) * 100, 2),
    'Win Rate (%)'      : round(results.get('win_rate', 0) * 100, 2),
    'Total Trades'      : results.get('total_trades', 'N/A'),
}
print('=== BACKTEST PERFORMANCE REPORT ===')
for k, v in metrics.items():
    print(f'  {k:<25}: {v}')

# ── Equity curve plot ────────────────────────────────────────────────────────
if hasattr(results, 'portfolio_value_trace'):
    fig, ax = plt.subplots(figsize=(12, 5), facecolor='#0d1117')
    ax.set_facecolor('#0d1117')

    portfolio_trace = results.portfolio_value_trace
    ax.plot(portfolio_trace.index, portfolio_trace.values,
            color='#00d4aa', linewidth=1.5, label='DeepScalper Portfolio')

    if hasattr(results, 'benchmark_value_trace'):
        bench = results.benchmark_value_trace
        ax.plot(bench.index, bench.values,
                color='#8b949e', linewidth=1.0, linestyle='--', label='SPY (Buy & Hold)')

    ax.set_title('DeepScalper Equity Curve vs SPY', color='white', fontsize=14)
    ax.set_xlabel('Date', color='#8b949e')
    ax.set_ylabel('Portfolio Value ($)', color='#8b949e')
    ax.tick_params(colors='#8b949e')
    ax.legend(facecolor='#161b22', labelcolor='white')
    ax.spines[:].set_color('#30363d')
    plt.tight_layout()
    plt.savefig('/content/equity_curve.png', dpi=150, bbox_inches='tight',
                facecolor='#0d1117')
    plt.show()
    print('Equity curve saved → /content/equity_curve.png')
else:
    print('Portfolio trace not available — run backtest with a Lumibot-compatible results object.')